 # Converting a grid model from LTDS to pandapower


Import the pandapower library and the neccessary methods for the conversion as follows:

In [ ]:
import os
import pandapower as pp
from pandapower.converter.cim import from_cim as cim2pp
from pandapower.converter.cim.cim_classes import CimParser
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

## LTDS to pandapower

First, we define the LTDS zip archive, which can be converted to pandapower. If there is no SSH profile available, there is an option to get some P and Q values from Excel. However, please take into account that there are some assumptions made, which can...

In [ ]:
# ltds_files is a list containing paths to files needed for the LTDS converter:
ltds_files = [r"path-to-ltds-zip"]
path_excel = [r"path-to-excel-input-files"]
excel_sheet_name = 'Table 3A - Load Data Observed'
excel_column_name = 'Maximum Demand 2023/24'

cim_parser = CimParser(cgmes_version='ltds')
cim_parser.parse_files(ltds_files).prepare_cim_net().set_cim_data_types()
cim = cim_parser.cim


excel_df = []
for one_excel_file in path_excel:
    if not os.path.isfile(one_excel_file):
        continue
    one_excel_df = pd.read_excel(one_excel_file, sheet_name=excel_sheet_name, skiprows=1)
    one_excel_df = one_excel_df.loc[one_excel_df['Season'] == 'Summer']
    one_excel_df = one_excel_df[['Sub-station', excel_column_name]]
    one_excel_df = one_excel_df.rename(columns={'Sub-station': 'name_excel', excel_column_name: 'p_mw_excel'})
    one_excel_df['p_mw_excel'] = one_excel_df['p_mw_excel'].astype(float)
    excel_df.append(one_excel_df)
if len(excel_df) > 0:
    excel_df = pd.concat(excel_df)
    excel_df = excel_df.drop_duplicates(subset=['name_excel'])
else:
    excel_df = pd.DataFrame(columns=['name_excel', 'p_mw_excel'])

cim['eq']['EnergyConsumer']['description'] = cim['eq']['EnergyConsumer']['description'].str.replace('11 kV', '11kV')

cim['eq']['EnergyConsumer']['description'] = cim['eq']['EnergyConsumer']['description'].astype(str)

cim['eq']['EnergyConsumer']['dups'] = cim['eq']['EnergyConsumer'].groupby('description')['description'].transform('count')

# add the loads to the SSH
cim['ssh']['EnergyConsumer'] = pd.concat([cim['ssh']['EnergyConsumer'], cim['eq']['EnergyConsumer'][['rdfId']]], ignore_index=True)
cim['ssh']['EnergyConsumer']['p'] = cim['eq']['EnergyConsumer']['description'].map(excel_df.set_index('name_excel')['p_mw_excel']) / cim['eq']['EnergyConsumer']['dups']
cim['ssh']['EnergyConsumer']['p'] = cim['ssh']['EnergyConsumer']['p'].fillna(0.)
cim['ssh']['EnergyConsumer']['q'] = cim['ssh']['EnergyConsumer']['q'].fillna(0.)
cim['ssh']['EnergyConsumer']['inService'] = cim['ssh']['EnergyConsumer']['inService'].fillna(True)

for one_sw in ['Breaker', 'Disconnector', 'Switch', 'LoadBreakSwitch']:
    cim['ssh'][one_sw] = pd.concat([cim['ssh'][one_sw], cim['eq'][one_sw][['rdfId', 'normalOpen']].rename(columns={'normalOpen': 'open'})], ignore_index=True)
    cim['ssh'][one_sw]['inService'] = True

for one_asset in ['ExternalNetworkInjection', 'ConformLoad', 'NonConformLoad', 'StationSupply',
                  'SynchronousMachine', 'AsynchronousMachine', 'EquivalentInjection', 'PowerElectronicsConnection']:
    cim['ssh'][one_asset] = pd.concat([cim['ssh'][one_asset], cim['eq'][one_asset][['rdfId']]], ignore_index=True)
    cim['ssh'][one_asset]['p'] = cim['ssh'][one_asset]['p'].fillna(0.)
    cim['ssh'][one_asset]['q'] = cim['ssh'][one_asset]['q'].fillna(0.)
    cim['ssh'][one_asset]['inService'] = cim['ssh'][one_asset]['inService'].fillna(True)

cim['ssh']['ExternalNetworkInjection']['referencePriority'] = cim['ssh']['ExternalNetworkInjection']['referencePriority'].fillna(1)
cim['ssh']['ExternalNetworkInjection']['controlEnabled'] = cim['ssh']['ExternalNetworkInjection']['controlEnabled'].fillna(True)
cim['ssh']['SynchronousMachine']['referencePriority'] = cim['ssh']['SynchronousMachine']['referencePriority'].fillna(0)
cim['ssh']['SynchronousMachine']['controlEnabled'] = cim['ssh']['SynchronousMachine']['controlEnabled'].fillna(False)
cim['ssh']['EquivalentInjection']['regulationStatus'] = cim['ssh']['EquivalentInjection']['regulationStatus'].fillna(False)

cim['ssh']['EnergySource'] = pd.concat([cim['ssh']['EnergySource'], cim['eq']['EnergySource'][['rdfId']]], ignore_index=True)
cim['ssh']['EnergySource']['activePower'] = cim['ssh']['EnergySource']['activePower'].fillna(0.)
cim['ssh']['EnergySource']['reactivePower'] = cim['ssh']['EnergySource']['reactivePower'].fillna(0.)
cim['ssh']['EnergySource']['inService'] = cim['ssh']['EnergySource']['inService'].fillna(True)

cim['ssh']['StaticVarCompensator'] = pd.concat([cim['ssh']['StaticVarCompensator'], cim['eq']['StaticVarCompensator'][['rdfId']]], ignore_index=True)
cim['ssh']['StaticVarCompensator']['q'] = cim['ssh']['StaticVarCompensator']['q'].fillna(0.)
cim['ssh']['StaticVarCompensator']['inService'] = cim['ssh']['StaticVarCompensator']['inService'].fillna(True)

# add the terminals
cim['ssh']['Terminal'] = pd.concat([cim['ssh']['Terminal'], cim['eq']['Terminal'][['rdfId']]], ignore_index=True)
cim['ssh']['Terminal']['connected'] = cim['ssh']['Terminal']['connected'].fillna(True)
# add the shunts
cim['ssh']['LinearShuntCompensator'] = pd.concat([cim['ssh']['LinearShuntCompensator'], cim['eq']['LinearShuntCompensator'][['rdfId', 'normalSections']].rename(
    columns={'normalSections': 'sections'})], ignore_index=True)
cim['ssh']['LinearShuntCompensator']['controlEnabled'] = cim['ssh']['LinearShuntCompensator']['controlEnabled'].fillna(False)
cim['ssh']['LinearShuntCompensator']['inService'] = cim['ssh']['LinearShuntCompensator']['inService'].fillna(True)
cim['ssh']['NonlinearShuntCompensator'] = pd.concat([cim['ssh']['NonlinearShuntCompensator'], cim['eq']['NonlinearShuntCompensator'][['rdfId']]], ignore_index=True)
cim['ssh']['NonlinearShuntCompensator']['sections'] = cim['ssh']['NonlinearShuntCompensator']['sections'].fillna(0)
cim['ssh']['NonlinearShuntCompensator']['controlEnabled'] = cim['ssh']['NonlinearShuntCompensator']['controlEnabled'].fillna(False)
cim['ssh']['NonlinearShuntCompensator']['inService'] = cim['ssh']['NonlinearShuntCompensator']['inService'].fillna(True)

# add the tap changer steps
cim['ssh']['RatioTapChanger'] = pd.concat([cim['ssh']['RatioTapChanger'], cim['eq']['RatioTapChanger'][['rdfId', 'neutralStep']].rename(
    columns={'neutralStep': 'step'})], ignore_index=True)
cim['ssh']['RatioTapChanger']['controlEnabled'] = cim['ssh']['RatioTapChanger']['controlEnabled'].fillna(False)
# add the TapChangerControls
cim['ssh']['TapChangerControl'] = pd.concat([cim['ssh']['TapChangerControl'], cim['eq']['TapChangerControl'][['rdfId']]], ignore_index=True)
cim['ssh']['TapChangerControl']['discrete'] = cim['ssh']['TapChangerControl']['discrete'].fillna(False)
cim['ssh']['TapChangerControl']['enabled'] = cim['ssh']['TapChangerControl']['enabled'].fillna(False)
cim['eq']['PowerTransformer']['inService'] = True

cim['ssh']['Equipment'] = pd.concat([cim['ssh']['Equipment'], cim['eq']['EquivalentBranch'][['rdfId']]], ignore_index=True)
cim['ssh']['Equipment']['inService'] = cim['ssh']['Equipment']['inService'].fillna(True)
cim['ssh']['Equipment'] = pd.concat([cim['ssh']['Equipment'], cim['eq']['ACLineSegment'][['rdfId']]], ignore_index=True)
cim['ssh']['Equipment']['inService'] = cim['ssh']['Equipment']['inService'].fillna(True)
cim['ssh']['Equipment'] = pd.concat([cim['ssh']['Equipment'], cim['eq']['DCLineSegment'][['rdfId']]], ignore_index=True)
cim['ssh']['Equipment']['inService'] = cim['ssh']['Equipment']['inService'].fillna(True)

# use the from_cim_dict to put in the modified CimParser
net = cim2pp.from_cim_dict(cim_parser=cim_parser, cim_version='LTDS', create_tap_controller=False)

# if there is no slack in the grid, create one
if net.gen.empty and net.ext_grid.empty and not net.sgen.empty:
    slack = net.sgen.loc[net.sgen.in_service].loc[net.sgen.p_mw == net.sgen.p_mw.max()]
    net.sgen = net.sgen.drop(slack.index[0])
    pp.create_gen(net, bus=slack.bus.iloc[0], p_mw=slack.p_mw.iloc[0], slack=True, in_service=True)

print('Conversion successful')

## Get an overview over your grid
Once the network is converted to pandapower, the data can be displayed:

In [ ]:
print(net)

## Export your grid
There are different options to export, for example as JSON, Excel or CSV:

In [ ]:
path_to_json = r'path-to-json.json'
path_to_excel = r'path-to-excel.xlsx'
path_to_csv = r'path-to-csv'
pp.to_json(net, path_to_json)
pp.to_excel(net, path_to_excel)
if os.path.isdir(path_to_csv):
    # for CSV, there is no pandapower method available
    for one_type in ['bus', 'line', 'impedance', 'trafo', 'trafo3w', 'load', 'sgen', 'gen']:
        net[one_type].to_csv(path_to_csv+'\\'+one_type+'.csv', sep=',')
else:
    print("Please provide a valid path for exporting the CSV data.")

## Display the nodes

In [ ]:
print(f"Overview over the nodes: {net.bus.describe()}")
print(f"The nodes: {net.bus.loc[:100]}")

## Display the lines

In [ ]:
print(f"Overview over the lines: {net.line.describe()}")
print(f"The lines: {net.line.loc[:100]}")

## Display the transformers

In [ ]:
# two winding transformers
print(f"Overview over the 2W transformers: {net.trafo.describe()}")
print(f"The 2W transformers: {net.trafo.loc[:100]}")
# three winding transformers
print(f"Overview over the 3W transformers: {net.trafo3w.describe()}")
print(f"The 3W transformers: {net.trafo3w.loc[:100]}")

## Display the loads

In [ ]:
print(f"Overview over the loads: {net.load.describe()}")
print(f"The loads: {net.load.loc[:100]}")

## Display the generation

In [ ]:
print(f"Overview over the PQ generators: {net.sgen.describe()}")
print(f"The PQ generators: {net.sgen.loc[:100]}")

print(f"Overview over the PV generators: {net.gen.describe()}")
print(f"The PV generators: {net.gen.loc[:100]}")

## Get only HV elements from the grid
In pandapower we are using pandas DataFrames, you can create your queries like you wish. Here is an example to get HV (110kV) elements from your grid.

In [ ]:
print("the HV nodes first")
print(net.bus.loc[(net.bus.vn_kv > 100) & (net.bus.vn_kv < 150)])

In [ ]:
print("now the HV lines")
net.line['vn_kv_bus'] = net.line.from_bus.map(net.bus.vn_kv)
print(net.line.loc[(net.line.vn_kv_bus > 100) & (net.line.vn_kv_bus < 150)])

In [ ]:
print("now the HV 2W trafos")
print(net.trafo.loc[(net.trafo.vn_hv_kv > 100) & (net.trafo.vn_hv_kv < 150)])

In [ ]:
print("now the HV 3W trafos")
print(net.trafo3w.loc[(net.trafo3w.vn_hv_kv > 100) & (net.trafo3w.vn_hv_kv < 150)])

In [ ]:
print("now the loads")
net.load['vn_kv_bus'] = net.load.bus.map(net.bus.vn_kv)
print(net.load.loc[(net.load.vn_kv_bus > 100) & (net.load.vn_kv_bus < 150)])

In [ ]:
print("now the PQ generators")
net.sgen['vn_kv_bus'] = net.sgen.bus.map(net.bus.vn_kv)
print(net.sgen.loc[(net.sgen.vn_kv_bus > 100) & (net.sgen.vn_kv_bus < 150)])

In [ ]:
print("now the PV generators")
net.gen['vn_kv_bus'] = net.gen.bus.map(net.bus.vn_kv)
print(net.gen.loc[(net.gen.vn_kv_bus > 100) & (net.gen.vn_kv_bus < 150)])